# Lab 01 — Build Your First Tool-Calling Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-01-build-your-first-tool-calling-agent/lab-01-build-your-first-tool-calling-agent.ipynb)

**Topic:** 1 — Modern Agent Foundations

**Objective:** Explain the components of a modern AI agent and implement a reasoning loop with tool calling

Build a single agent from first principles so the loop is not a black box: the model receives a goal, decides to call a tool, your code executes it, and the result is fed back until the agent can answer.

Full step-by-step instructions are in the Learner Guide.


In [ ]:
!pip install -q openai python-dotenv


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("OPENAI_API_KEY")
print("Keys set:", [v for v in ["OPENAI_API_KEY"] if os.environ.get(v)])


## 1. Imports and the client

The key is read from the environment, never from source.


In [ ]:
import json
import os

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"


## 2. Two plain Python functions the agent may call

These are ordinary functions with no AI in them at all — that is the point. The agent decides *when* to call them; your code decides *what they do*.


In [ ]:
def get_weather(city: str) -> dict:
    """Return the current weather for a city (stubbed for the lab)."""
    fake_readings = {
        "singapore": {"temp_c": 31, "condition": "Thundery showers", "humidity": 84},
        "london": {"temp_c": 12, "condition": "Overcast", "humidity": 71},
        "tokyo": {"temp_c": 18, "condition": "Clear", "humidity": 55},
    }
    reading = fake_readings.get(city.strip().lower())
    if reading is None:
        return {"error": f"No weather data for '{city}'."}
    return {"city": city, **reading}


def calculate(expression: str) -> dict:
    """Evaluate a simple arithmetic expression such as '0.15 * 2400'."""
    allowed = set("0123456789.+-*/() ")
    if not set(expression) <= allowed:
        return {"error": "Expression contains unsupported characters."}
    try:
        # Safe because the character set above permits arithmetic only.
        result = eval(expression, {"__builtins__": {}}, {})
    except Exception as exc:
        return {"error": f"Could not evaluate: {exc}"}
    return {"expression": expression, "result": result}


## 3. Describe both functions as JSON tool schemas

The model never sees your Python. It sees only this JSON description, so the `description` fields are what actually drive the routing decision. Vague descriptions produce a confused agent.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": (
                "Get the current weather for a named city. Use this whenever the "
                "user asks about weather, temperature or conditions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, for example 'Singapore'.",
                    }
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": (
                "Evaluate an arithmetic expression. Use this for any calculation "
                "instead of doing the arithmetic yourself."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Arithmetic expression, e.g. '0.15 * 2400'.",
                    }
                },
                "required": ["expression"],
            },
        },
    },
]

# Maps the schema name back to the real Python function.
TOOL_REGISTRY = {
    "get_weather": get_weather,
    "calculate": calculate,
}


## 4. The agent loop

The heart of the lab. Send the messages plus the tool schemas; if the reply contains `tool_calls`, execute each matching function and append the result as a `tool` message; then loop so the model can reason over what it just learned.

Note the ordering rule the API enforces: every `tool` message must follow the assistant message that requested it, and must carry the matching `tool_call_id`. The `max_turns` cap is a safety requirement, not a nicety.


In [ ]:
def run_agent(user_question: str, max_turns: int = 5) -> str:
    """Run the reason-act-observe loop until the model produces a final answer."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Use the supplied tools rather than "
                "guessing. When you have everything you need, answer in plain prose."
            ),
        },
        {"role": "user", "content": user_question},
    ]

    for turn in range(1, max_turns + 1):
        print(f"\n--- Turn {turn} ---")
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOLS,
        )
        message = response.choices[0].message

        # No tool calls means the agent is ready to answer.
        if not message.tool_calls:
            print("Agent produced a final answer.")
            return message.content

        # Append the assistant turn that requested the tools, then each result.
        messages.append(message)

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            print(f"  Tool call: {name}({arguments})")

            function = TOOL_REGISTRY.get(name)
            if function is None:
                result = {"error": f"Unknown tool '{name}'."}
            else:
                try:
                    result = function(**arguments)
                except Exception as exc:
                    result = {"error": f"Tool raised: {exc}"}

            print(f"  Tool result: {result}")
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result),
                }
            )

    return "Stopped: reached the maximum number of turns without a final answer."


## 5. Run the agent

You should see two turns: the first requesting both tools, the second producing prose.


In [ ]:
answer = run_agent("What is the weather in Singapore, and what is 15% of 2400?")
print("\n=== Final answer ===")
print(answer)


## 6. Test the failure path

An unknown city returns the tool's error dictionary and the agent should report it rather than crashing.


In [ ]:
print(run_agent("What is the weather in Atlantis?"))
